# Exploration & Nettoyage — Risk Monitor Dataset

**Objectif** : Explorer, comprendre et nettoyer la base de données `risk_monitor_dataset.sqlite`  
**Tables** : `users`, `subscriptions`, `memberships`, `payments`, `complaints`  
**Contrainte** : Aucun dictionnaire de données fourni — les anomalies et les codes de statut sont à déduire des données.

---
## Plan
1. Chargement des données
2. Exploration de chaque table
3. Nettoyage de chaque table
4. Reverse-engineering des codes `status`
5. Sauvegarde des tables nettoyées

---
## 1. Chargement des données

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Connexion à la base de données SQLite
conn = sqlite3.connect("../data/risk_monitor_dataset.sqlite")

# Vérification des tables disponibles
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)

,name
0,users
1,subscriptions
2,memberships
3,payments
4,complaints


In [3]:
# Chargement de toutes les tables en DataFrames
users         = pd.read_sql("SELECT * FROM users", conn)
subscriptions = pd.read_sql("SELECT * FROM subscriptions", conn)
memberships   = pd.read_sql("SELECT * FROM memberships", conn)
payments      = pd.read_sql("SELECT * FROM payments", conn)
complaints    = pd.read_sql("SELECT * FROM complaints", conn)

print(f"users         : {len(users)} lignes")
print(f"subscriptions : {len(subscriptions)} lignes")
print(f"memberships   : {len(memberships)} lignes")
print(f"payments      : {len(payments)} lignes")
print(f"complaints    : {len(complaints)} lignes")

users         : 2001 lignes
subscriptions : 400 lignes
memberships   : 1083 lignes
payments      : 7277 lignes
complaints    : 1213 lignes


---
## 2. Exploration et Nettoyage de la Table `users`

### 2.1 Exploration initiale

In [4]:
# Structure générale
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 2001 entries, 0 to 2000
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             2001 non-null   int64  
 1   email          2001 non-null   str    
 2   country        2001 non-null   str    
 3   signup_date    2001 non-null   str    
 4   status         1976 non-null   float64
 5   last_seen      2001 non-null   str    
 6   referral_code  583 non-null    str    
 7   phone_prefix   1582 non-null   str    
dtypes: float64(1), int64(1), str(6)
memory usage: 249.1 KB


In [ ]:
# Valeurs manquantes par colonne
users.isnull().sum()

id                  0
email               0
country             0
signup_date         0
status             25
last_seen           0
referral_code    1418
phone_prefix      419
dtype: int64

In [ ]:
# status : 25 NaN 
# referral_code : 1418 NaN ==> la majorité n'a pas de parrain 
#  phone_prefix : 419 NaN

In [7]:
# Doublons exacts
print("Doublons exacts :", users.duplicated().sum())

# Email dupliqué : un même email pour deux users différents ?
print("Emails dupliqués :", users["email"].duplicated().sum())

Doublons exacts : 0
Emails dupliqués : 1


In [ ]:
# Aperçu des données brute:
users.head()

,id,email,country,signup_date,status,last_seen,referral_code,phone_prefix
0,1,user_1@yahoo.fr,IT,2021-02-09 21:26:47,1.0,2021-04-28 14:23:00,NaN,+49
1,2,user_2@gmail.com,CH,2022-01-09T06:49:44Z,0.0,2023-01-05 17:03:20,NaN,+41
2,3,user_3@outlook.com,BE,1584765297,1.0,2021-07-30 05:38:37,27cb14dc,+32
3,4,user_4@hotmail.com,DE,2020-07-13T23:53:58Z,0.0,2024-06-10 09:06:04,NaN,+49
4,5,user_5@outlook.com,DE,21/07/2020 08:54,0.0,2026-03-06 20:16:50,NaN,+49


In [9]:
# on remarque plusieurs problèmes :
# signup_date :  formats différents 
# phone_prefix : IT ET DE avec le meme prefixe (+49 pour un user IT qui devrait être +39) 

In [10]:
# Distribution du champ status (codes non documentés)
users["status"].value_counts(dropna=False).sort_index()

status
-1.0       38
 0.0     1325
 1.0      179
 2.0      107
 3.0      110
 4.0      198
 99.0      19
 NaN       25
Name: count, dtype: int64

In [11]:
# Anomalies   25 NaN
# on a diff valeurs pour le status ( -1 et 99 devrai representer une anomalie)

In [ ]:
# Distribution du champ country
users["country"].value_counts()

country
ES        165
AT        153
DE        149
IT        140
BE        135
France    134
fr        134
PT        133
          132
Fra       127
FR        127
DE        122
NL        121
 ES       119
CH        110
Name: count, dtype: int64

In [ ]:
# formats incohérents : 'France', 'fr', 'Fra', ' ES' , ' '

In [ ]:
# Validation email
print("Emails valides (contiennent @) :", users["email"].str.contains("@").mean())

Emails valides (contiennent @) : 1.0


In [ ]:
#format email correcte:  tous les emails contiennent bien un '@'

### 2.2 Nettoyage — `country`

**Problème** : formats variés — `France`, `fr`, `Fra`, ` ES` (avec espace), `DE` en double, champ vide  
**Décision** : normaliser en codes ISO pour chaque pays,  2 lettres majuscules via un mapping après strip + lower

In [ ]:
country_map = {
    "fr": "FR", "fra": "FR", "france": "FR",
    "es": "ES", "de": "DE", "it": "IT",
    "be": "BE", "nl": "NL", "ch": "CH",
    "pt": "PT", "at": "AT"
}

users["country"] = (
    users["country"]
    .astype(str)
    .str.strip()          # supprime les espaces avant/après
    .str.lower()          # tout en minuscule pour matcher les clés du mapping
    .replace(country_map) # normalise les variantes connues
    .str.upper()          # repasse en majuscule
)

# 132 users restent avec country vide — on ne peut pas déduire leur pays
print(users["country"].value_counts())

country
FR    522
ES    284
DE    271
AT    153
IT    140
BE    135
PT    133
      132
NL    121
CH    110
Name: count, dtype: int64


In [21]:
users[users["country"] == "IT"]["phone_prefix"].value_counts()

phone_prefix
+39     128
+49       2
+34       2
+32       2
+43       2
+33       1
+351      1
+31       1
+41       1
Name: count, dtype: int64

In [22]:
# Vérifier la cohérence pour TOUS les pays
phone_country_map = {
    "FR": "+33", "DE": "+49", "IT": "+39",
    "ES": "+34", "BE": "+32", "NL": "+31",
    "CH": "+41", "PT": "+351", "AT": "+43"
}

for country, expected in phone_country_map.items():
    subset = users[users["country"] == country]["phone_prefix"].value_counts()
    total = len(users[users["country"] == country])
    correct = subset.get(expected, 0)
    wrong = total - correct
    print(f"{country} ({total} users) → {correct} correct ({expected}), {wrong} mauvais préfixe")

FR (522 users) → 145 correct (+33), 377 mauvais préfixe
DE (271 users) → 122 correct (+49), 149 mauvais préfixe
IT (140 users) → 128 correct (+39), 12 mauvais préfixe
ES (284 users) → 163 correct (+34), 121 mauvais préfixe
BE (135 users) → 124 correct (+32), 11 mauvais préfixe
NL (121 users) → 112 correct (+31), 9 mauvais préfixe
CH (110 users) → 99 correct (+41), 11 mauvais préfixe
PT (133 users) → 125 correct (+351), 8 mauvais préfixe
AT (153 users) → 145 correct (+43), 8 mauvais préfixe


In [23]:
# Vérifier ce que contiennent les "mauvais" préfixes pour FR et DE
print("=== Préfixes pour FR ===")
print(users[users["country"] == "FR"]["phone_prefix"].value_counts())

print("\n=== Préfixes pour DE ===")
print(users[users["country"] == "DE"]["phone_prefix"].value_counts())

=== Préfixes pour FR ===
phone_prefix
+33     145
+39      26
+41      26
+351     25
+43      25
+32      19
+34      18
+31      18
+49      16
Name: count, dtype: int64

=== Préfixes pour DE ===
phone_prefix
+49     122
+41      15
+31      12
+351     10
+39       9
+32       7
+34       7
+43       6
+33       5
Name: count, dtype: int64


### 2.3 Nettoyage — `signup_date` et `last_seen`
**Probleme**
Apres une recherche sur les format de date qui existe dans la bdd j'ai reussi a trouver 4 4 formats de dates coexistent dans `signup_date` :

- datetime standard : `2021-02-09 21:26:47`
- ISO 8601 avec Z : `2022-01-09T06:49:44Z`
- timestamp Unix : `1584765297`
- format français : `21/07/2020 08:54`

**Décision** : parser chaque format et convertir en UTC

In [15]:
def parse_signup_date(val):
    """Gère les 4 formats de dates présents dans signup_date."""
    try:
        if str(val).strip().isdigit():  # timestamp Unix
            return pd.Timestamp(int(val), unit='s', tz='UTC')
        return pd.to_datetime(val, dayfirst=True, utc=True)
    except Exception:
        return pd.NaT

users["signup_date"] = users["signup_date"].apply(parse_signup_date)
users["last_seen"]   = pd.to_datetime(users["last_seen"], errors="coerce", utc=True)

print(f"signup_date NaT : {users['signup_date'].isna().sum()}")
print(f"last_seen   NaT : {users['last_seen'].isna().sum()}")

C:\Users\Assia\AppData\Local\Temp\ipykernel_23060\425988093.py:6: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(val, dayfirst=True, utc=True)
C:\Users\Assia\AppData\Local\Temp\ipykernel_23060\425988093.py:6: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(val, dayfirst=True, utc=True)


signup_date NaT : 0
last_seen   NaT : 0


In [17]:
# verifecation de lexistance de l Incohérence entre  last_seen et  signup_date
# verifier que tout les last_seen sont apres  signup_date
mask_inv = users["last_seen"] < users["signup_date"]
print(f"Lignes avec last_seen < signup_date : {mask_inv.sum()}")


Lignes avec last_seen < signup_date : 0


In [18]:
# Incohérence : last_seen avant signup_date
# Hypothèse : les deux colonnes sont inversées pour ces lignes → on les swap
users.loc[mask_inv, ["signup_date", "last_seen"]] = (
    users.loc[mask_inv, ["last_seen", "signup_date"]].values
)
print(f"Incohérences restantes : {(users['last_seen'] < users['signup_date']).sum()}")

Incohérences restantes : 0


In [19]:
# last_seen dans le futur : impossible physiquement
# Décision : on ne peut pas deviner la vraie date → on remplace par NaT et on flag
now = pd.Timestamp.now(tz="UTC")
users["last_seen_future"] = (users["last_seen"] > now).astype(int)
print(f"last_seen dans le futur : {users['last_seen_future'].sum()}")



last_seen dans le futur : 11


In [20]:
users.loc[users["last_seen"] > now, "last_seen"] = pd.NaT

### 2.4 Nettoyage — `phone_prefix`

**Problème** : les préfixes téléphoniques ne correspondent pas au pays déclaré  
**Investigation** : pour chaque pays, le préfixe majoritaire correspond à l'indicatif officiel.  
Les outliers sont distribués aléatoirement sur tous les autres pays 
**Décision** : le `country` est fiable, on remplace `phone_prefix` par le préfixe officiel du pays

In [25]:
phone_country_map = {
    "FR": "+33", "DE": "+49", "IT": "+39",
    "ES": "+34", "BE": "+32", "NL": "+31",
    "CH": "+41", "PT": "+351", "AT": "+43"
}


# Correction : on remplace par le préfixe officiel du pays
users["phone_prefix_raw"] = users["phone_prefix"].copy()  # on garde l'original
users["phone_prefix"]     = users["country"].map(phone_country_map)

users["phone_missing"] = users["phone_prefix"].isna().astype(int)
print(f"\nphone_prefix NaN (country vide) : {users['phone_missing'].sum()}")


phone_prefix NaN (country vide) : 132


### 2.5 Nettoyage — `status`

**Problème** : type float au lieu d'entier, 25 NaN, code 99 suspect  
**Décision** : convertir en Int64 nullable, flag les NaN et les 99

In [26]:
# Flag avant conversion
users["status_missing"] = users["status"].isna().astype(int)
users["status_anomaly"] = users["status"].isin([99]).astype(int)

# Conversion en Int64 nullable (pas float) car c'est un code catégoriel
users["status"] = users["status"].astype("Int64")

print(users["status"].value_counts(dropna=False).sort_index())
print(f"\nstatus manquant : {users['status_missing'].sum()}")
print(f"status=99 (anomalie) : {users['status_anomaly'].sum()}")

status
-1        38
0       1325
1        179
2        107
3        110
4        198
99        19
<NA>      25
Name: count, dtype: Int64

status manquant : 25
status=99 (anomalie) : 19


### 2.6 Nettoyage — email dupliqué

**Problème** : `user_26@proton.me` apparaît pour deux users (id=26 et id=1834)  
**Investigation** : id=1834 a des memberships, payments, complaints et une subscription. id=26 n'a aucune activité.  
**Décision** : flag id=26 comme doublon, id=1834 est le vrai compte

In [27]:
# Identifier le doublon
print(users[users["email"].duplicated(keep=False)][["id", "email", "signup_date", "country"]])


        id              email               signup_date country
25      26  user_26@proton.me 2023-04-23 01:56:13+00:00      FR
1833  1834  user_26@proton.me 2022-02-26 03:01:15+00:00      AT


In [28]:
# Les deux users avec le même email
dup = users[users["email"] == "user_26@proton.me"]
print(dup[["id", "email", "signup_date", "status", "country"]])

id1, id2 = dup["id"].values  # id1=26, id2=1834
print(f"\nid1={id1}, id2={id2}")

# Est-ce qu'ils ont des memberships ?
print("\n table memebership")
print(memberships[memberships["user_id"].isin([id1, id2])])

# Est-ce qu'ils ont des payments ?
print("\n table payments ")
print(payments[payments["user_id"].isin([id1, id2])])

# Est-ce qu'ils ont des complaints ?
print("\n table complaints ")
print(complaints[complaints["reporter_id"].isin([id1, id2]) | complaints["target_id"].isin([id1, id2])])

# Est-ce qu'ils ont des subscriptions ?
print("\n table subscriptions ")
print(subscriptions[subscriptions["owner_id"].isin([id1, id2])])

        id              email               signup_date  status country
25      26  user_26@proton.me 2023-04-23 01:56:13+00:00       0      FR
1833  1834  user_26@proton.me 2022-02-26 03:01:15+00:00       0      AT

id1=26, id2=1834

 table memebership
      id  user_id  subscription_id  status            joined_at left_at reason
191  886     1834              348       5  2025-05-13 09:57:57     NaN    NaN

 table payments 
        id  user_id  subscription_id  amount_cents  fee_cents     status  \
1269  1270     1834              348           869        102  Succeeded   

                     created_at          captured_at currency  \
1269  2025-05-10T09:57:57+01:00  2025-05-12 15:57:57      EUR   

     stripe_error_code  
1269               NaN  

 table complaints 
      id  reporter_id  target_id  subscription_id                type  \
403  404         1834       1547              348  owner_unresponsive   
737  738         1834         19              348               other 

In [29]:
# Flag : id=26 est le compte sans activité
users["is_duplicate_email"] = (users["id"] == 26).astype(int)
print(f"\nDoublons email flagués : {users['is_duplicate_email'].sum()}")


Doublons email flagués : 1


### 2.7 Flags utiles

Ces colonnes seront utilisées dans le scoring de risque

In [30]:
users["has_referral"] = users["referral_code"].notna().astype(int)

print(f"has_referral : {users['has_referral'].sum()} users parrainés")

has_referral : 583 users parrainés


### 2.8 Résumé du nettoyage — `users`

In [31]:
print("=== USERS APRÈS NETTOYAGE ===")
print(f"Lignes totales           : {len(users)}")
print(f"signup_date NaT          : {users['signup_date'].isna().sum()}")
print(f"last_seen NaT            : {users['last_seen'].isna().sum()}")
print(f"last_seen dans le futur  : {users['last_seen_future'].sum()} (remplacés par NaT)")
print(f"status NaN               : {users['status'].isna().sum()}")
print(f"status=99 (anomalie)     : {users['status_anomaly'].sum()}")
print(f"email dupliqué           : {users['is_duplicate_email'].sum()}")
print(f"country vide             : {(users['country'] == '').sum()}")
print(f"\nPays disponibles : {sorted(users['country'].unique())}")
users.dtypes

=== USERS APRÈS NETTOYAGE ===
Lignes totales           : 2001
signup_date NaT          : 0
last_seen NaT            : 11
last_seen dans le futur  : 11 (remplacés par NaT)
status NaN               : 25
status=99 (anomalie)     : 19
email dupliqué           : 1
country vide             : 132

Pays disponibles : ['', 'AT', 'BE', 'CH', 'DE', 'ES', 'FR', 'IT', 'NL', 'PT']


id                                  int64
email                                 str
country                               str
signup_date           datetime64[us, UTC]
status                              Int64
last_seen             datetime64[us, UTC]
referral_code                         str
phone_prefix                          str
last_seen_future                    int64
phone_prefix_raw                      str
phone_missing                       int64
status_missing                      int64
status_anomaly                      int64
is_duplicate_email                  int64
has_referral                        int64
dtype: object

---
## 3. Exploration et Nettoyage de la Table `subscriptions`

### 3.1 Exploration initiale

In [32]:
subscriptions.info()
print(subscriptions.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           400 non-null    int64
 1   brand        400 non-null    str  
 2   owner_id     400 non-null    int64
 3   created_at   400 non-null    str  
 4   status       400 non-null    int64
 5   max_slots    400 non-null    int64
 6   price_cents  400 non-null    int64
 7   currency     400 non-null    str  
dtypes: int64(5), str(3)
memory usage: 37.1 KB
id             0
brand          0
owner_id       0
created_at     0
status         0
max_slots      0
price_cents    0
currency       0
dtype: int64


In [33]:
subscriptions.head()

,id,brand,owner_id,created_at,status,max_slots,price_cents,currency
0,1,Microsoft 365,1,2023-01-14 23:23:57,0,4,1070,eur
1,2,HBO Max,1670,2023-07-16 21:39:51,2,4,233,USD
2,3,ChatGPT Plus,558,2024-10-02 13:54:25,0,3,1367,EUR
3,4,Microsoft 365,567,2023-10-26 23:03:50,0,6,460,EUR
4,5,NordVPN,950,2024-11-30 08:35:40,0,4,464,EUR


In [34]:
print("=== status ===")
print(subscriptions["status"].value_counts().sort_index())

print("\n=== currency ===")
print(subscriptions["currency"].value_counts())

print("\n=== brands ===")
print(subscriptions["brand"].value_counts())

=== status ===
status
0    229
1     46
2     68
3     57
Name: count, dtype: int64

=== currency ===
currency
EUR    294
eur     40
        31
€       19
USD     16
Name: count, dtype: int64

=== brands ===
brand
Midjourney         36
Microsoft 365      33
HBO Max            31
Disney+            31
Spotify            30
ChatGPT Plus       28
Apple Music        28
NordVPN            25
Notion             25
Adobe CC           24
Netflix            23
Canal+             23
YouTube Premium    22
Deezer             22
Amazon Prime       19
Name: count, dtype: int64


In [35]:
# Vérifier les owner_id orphelins (users qui n'existent pas)
orphans = subscriptions[~subscriptions["owner_id"].isin(users["id"])]
print(f"owner_id introuvables dans users : {len(orphans)}")
print(orphans[["id", "brand", "owner_id", "status", "price_cents"]])

owner_id introuvables dans users : 10
      id            brand  owner_id  status  price_cents
13    14    Microsoft 365      2265       3         1355
34    35           Notion      2390       2          838
88    89           Notion      2475       0         1060
93    94       Midjourney      2145       0         1470
173  174  YouTube Premium      2243       3         1062
188  189           Canal+      2409       0          549
287  288          NordVPN      2407       0         1473
338  339           Canal+      2404       0           15
342  343  YouTube Premium      2234       0         1166
392  393          Disney+      2087       3          631


In [40]:
orphan_ids = orphans["owner_id"].tolist()

print("=== memberships ===")
print(memberships[memberships["user_id"].isin(orphan_ids)])

print("\n=== payments ===")
print(payments[payments["user_id"].isin(orphan_ids)])

print("\n=== complaints ===")
print(complaints[complaints["reporter_id"].isin(orphan_ids) | 
                  complaints["target_id"].isin(orphan_ids)])

=== memberships ===
Empty DataFrame
Columns: [id, user_id, subscription_id, status, joined_at, left_at, reason]
Index: []

=== payments ===
Empty DataFrame
Columns: [id, user_id, subscription_id, amount_cents, fee_cents, status, created_at, captured_at, currency, stripe_error_code]
Index: []

=== complaints ===
Empty DataFrame
Columns: [id, reporter_id, target_id, subscription_id, type, status, created_at, resolved_at, resolution]
Index: []


### 3.2 Nettoyage — `currency`

**Problème** : casse incohérente (`EUR`, `eur`), champ vide, symbole `€`  
**Investigation** : les subscriptions avec currency vide appartiennent toutes à des owners européens  
**Décision** : normaliser en majuscules, remplacer vide et `€` par `EUR`

In [36]:
subscriptions["currency"] = (
    subscriptions["currency"]
    .astype(str)
    .str.strip()
    .replace({"eur": "EUR", "€": "EUR", "": "EUR"})
)

print(subscriptions["currency"].value_counts())

currency
EUR    384
USD     16
Name: count, dtype: int64


### 3.3 Nettoyage — `created_at`

In [37]:
subscriptions["created_at"] = pd.to_datetime(
    subscriptions["created_at"], errors="coerce", utc=True
)
print(f"created_at NaT : {subscriptions['created_at'].isna().sum()}")

now = pd.Timestamp.now(tz="UTC")
print(f"created_at dans le futur : {(subscriptions['created_at'] > now).sum()}")

created_at NaT : 0
created_at dans le futur : 0


### 3.4 Nettoyage — `owner_id` orphelins

**Problème** : 10 subscriptions ont un owner_id qui n'existe pas dans users (IDs > 2001)  
**Investigation** : aucune activité associée (0 memberships, 0 payments, 0 complaints)  
**Décision** : flag avec `owner_missing=1` — probablement des comptes supprimés

In [41]:
subscriptions["owner_missing"] = (
    ~subscriptions["owner_id"].isin(users["id"])
).astype(int)

print(f"Subscriptions sans owner valide : {subscriptions['owner_missing'].sum()}")

Subscriptions sans owner valide : 10


### 3.5 Note — `price_cents`

**Observation** : des prix vont de 2 à 1470 centimes pour le même service (ex: Midjourney).  
**Conclusion** : on ne peut pas identifier les aberrations sans référentiel externe.  
**Décision** : `price_cents` est considéré non fiable. On utilisera `payments.amount_cents` pour les analyses de revenus.

In [39]:
print(subscriptions["price_cents"].describe())
print("\nExemple Midjourney (même service, prix très variés) :")
print(subscriptions[subscriptions["brand"] == "Midjourney"]["price_cents"].describe())

count     400.00000
mean      793.94750
std       408.95539
min         2.00000
25%       498.00000
50%       811.50000
75%      1119.00000
max      1496.00000
Name: price_cents, dtype: float64

Exemple Midjourney (même service, prix très variés) :
count      36.000000
mean      781.722222
std       425.517944
min         6.000000
25%       504.500000
50%       841.500000
75%      1099.000000
max      1470.000000
Name: price_cents, dtype: float64


---
## 4. Exploration et Nettoyage de la Table `memberships`

### 4.1 Exploration initiale

In [42]:
memberships.info()
print(memberships.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1083 entries, 0 to 1082
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id               1083 non-null   int64
 1   user_id          1083 non-null   int64
 2   subscription_id  1083 non-null   int64
 3   status           1083 non-null   int64
 4   joined_at        1083 non-null   str  
 5   left_at          446 non-null    str  
 6   reason           377 non-null    str  
dtypes: int64(4), str(3)
memory usage: 90.9 KB
id                   0
user_id              0
subscription_id      0
status               0
joined_at            0
left_at            637
reason             706
dtype: int64


In [43]:
memberships.head()

,id,user_id,subscription_id,status,joined_at,left_at,reason
0,563,1485,231,1,2024-12-06 21:23:01,NaN,NaN
1,1042,690,399,2,2024-09-23 02:25:46,2025-03-31 22:56:44,fraud
2,72,56,31,1,2025-02-15 18:19:15,NaN,NaN
3,414,590,168,1,2024-08-06 15:33:32,NaN,NaN
4,413,20,168,1,2024-05-19 10:34:05,NaN,NaN


In [44]:
# Distribution status memberships
memberships["status"].value_counts().sort_index()

status
0     59
1    428
2    218
3    107
4     64
5     84
6     68
7     55
Name: count, dtype: int64

### 4.2 Nettoyage — dates

In [45]:
memberships["joined_at"] = pd.to_datetime(memberships["joined_at"], errors="coerce", utc=True)
memberships["left_at"]   = pd.to_datetime(memberships["left_at"],   errors="coerce", utc=True)

print(f"joined_at NaT : {memberships['joined_at'].isna().sum()}")
print(f"left_at   NaT : {memberships['left_at'].isna().sum()}")

# Vérifications de cohérence
print(f"left_at avant joined_at : {(memberships['left_at'] < memberships['joined_at']).sum()}")

now = pd.Timestamp.now(tz="UTC")
print(f"joined_at dans le futur : {(memberships['joined_at'] > now).sum()}")
print(f"left_at dans le futur   : {(memberships['left_at'] > now).sum()}")

joined_at NaT : 0
left_at   NaT : 637
left_at avant joined_at : 0
joined_at dans le futur : 0
left_at dans le futur   : 0


### 4.3 Nettoyage — `reason` vide

**Problème** : 70 lignes ont une chaîne vide `""` au lieu de NaN — non détectées par `isnull()`  
**Décision** : remplacer par NaN

In [46]:
memberships["reason"] = memberships["reason"].replace("", None)
print(memberships["reason"].value_counts(dropna=False))

reason
NaN               776
voluntary          69
owner_request      67
fraud              62
payment_failed     56
inactive           53
Name: count, dtype: int64


### 4.4 Nettoyage — `subscription_id` orphelins

**Problème** : 15 memberships pointent vers des subscription_id qui n'existent pas  
**Décision** : flag avec `subscription_missing=1`

In [47]:
memberships["subscription_missing"] = (
    ~memberships["subscription_id"].isin(subscriptions["id"])
).astype(int)

print(f"subscription_id orphelins : {memberships['subscription_missing'].sum()}")

subscription_id orphelins : 15


### 4.5 Anomalie — status=1 avec `left_at` renseigné

**Problème** : 14 membres avec status=1 (actif hypothétique) ont une date de départ  
**Hypothèse** : le status n'a pas été mis à jour quand le membre est parti  
**Décision** : flag avec `status_left_incoherent=1`

In [48]:
mask_incoherent = (memberships["status"] == 1) & (memberships["left_at"].notna())
memberships["status_left_incoherent"] = mask_incoherent.astype(int)

print(f"Status=1 avec left_at renseigné : {mask_incoherent.sum()}")

Status=1 avec left_at renseigné : 14


---
## 5. Exploration et Nettoyage de la Table `payments`

### 5.1 Exploration initiale

In [49]:
payments.info()
print(payments.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 7277 entries, 0 to 7276
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 7277 non-null   int64
 1   user_id            7277 non-null   int64
 2   subscription_id    7277 non-null   int64
 3   amount_cents       7277 non-null   int64
 4   fee_cents          7277 non-null   int64
 5   status             7277 non-null   str  
 6   created_at         7277 non-null   str  
 7   captured_at        3907 non-null   str  
 8   currency           7277 non-null   str  
 9   stripe_error_code  1833 non-null   str  
dtypes: int64(5), str(5)
memory usage: 882.3 KB
id                      0
user_id                 0
subscription_id         0
amount_cents            0
fee_cents               0
status                  0
created_at              0
captured_at          3370
currency                0
stripe_error_code    5444
dtype: int64


In [ ]:
print(payments["status"].value_counts(dropna=False).sort_index())

print("\n=== currency ===")
print(payments["currency"].value_counts())

print("\n=== amount_cents ===")
print(payments["amount_cents"].describe())

status
FAILED        210
Succeeded     232
canceled      141
disputed      216
failed       1774
pending       581
refunded      375
succeeded    3322
success       216
suceeded      210
Name: count, dtype: int64

=== currency ===
currency
EUR    5437
eur     741
        570
USD     380
GBP     149
Name: count, dtype: int64

=== amount_cents ===
count    7.277000e+03
mean     2.827295e+04
std      5.235191e+05
min     -1.499000e+03
25%      4.950000e+02
50%      8.250000e+02
75%      1.169000e+03
max      9.999900e+06
Name: amount_cents, dtype: float64


In [51]:
# Status : 10 variantes pour ~6 états réels (problème de casse + typo 'suceeded')

In [52]:
# Anomalie : user_id=9999 avec 20 paiements à 99 999€ → user de test
print("Top 10 montants :")
print(payments.nlargest(10, "amount_cents")[["user_id", "subscription_id", "amount_cents", "status"]])

# Orphelins
print(f"\nuser_id orphelins : {(~payments['user_id'].isin(users['id'])).sum()}")
print(f"subscription_id orphelins : {(~payments['subscription_id'].isin(subscriptions['id'])).sum()}")

Top 10 montants :
      user_id  subscription_id  amount_cents     status
7256     9999              348       9999900  succeeded
7257     9999              139       9999900  succeeded
7258     9999              112       9999900  succeeded
7259     9999               69       9999900  succeeded
7260     9999              206       9999900  succeeded
7261     9999               50       9999900  succeeded
7262     9999              274       9999900  succeeded
7263     9999              192       9999900  succeeded
7264     9999              368       9999900  succeeded
7265     9999              296       9999900  succeeded

user_id orphelins : 30
subscription_id orphelins : 121


### 5.2 Nettoyage — dates

In [53]:
payments["created_at"]  = pd.to_datetime(payments["created_at"],  errors="coerce", utc=True)
payments["captured_at"] = pd.to_datetime(payments["captured_at"], errors="coerce", utc=True)

print(f"created_at NaT  : {payments['created_at'].isna().sum()}")
print(f"captured_at NaT : {payments['captured_at'].isna().sum()} (normal — absent sur failed/pending)")
print(f"captured_at avant created_at : {(payments['captured_at'] < payments['created_at']).sum()}")

created_at NaT  : 6196
captured_at NaT : 3370 (normal — absent sur failed/pending)
captured_at avant created_at : 0


### 5.3 Nettoyage — `amount_cents` négatifs

**Problème** : des montants négatifs pour des status `refunded` et `disputed` 
**Analyse** : c'est une convention comptable — le signe négatif indique un remboursement  
**Décision** : valeur absolue — le `status` porte déjà l'information "remboursement"

In [55]:
print(f"Montants négatifs : {(payments['amount_cents'] < 0).sum()}")

Montants négatifs : 233


In [54]:
print(payments[payments["amount_cents"] < 0][["user_id", "amount_cents", "status", "stripe_error_code"]])

      user_id  amount_cents    status stripe_error_code
24        590         -1041  refunded               NaN
45        513          -338  refunded               NaN
49        372          -530  disputed               NaN
78        999         -1331  refunded               NaN
83        999          -429  refunded               NaN
...       ...           ...       ...               ...
6894       80          -441  refunded               NaN
6904       80         -1023  disputed               NaN
6930     1858          -670  refunded               NaN
6953       97          -562  disputed               NaN
7188      175         -1253  refunded               NaN

[233 rows x 4 columns]


In [56]:
payments["amount_cents"] = payments["amount_cents"].abs()
payments["fee_cents"]    = payments["fee_cents"].abs()

print(f"\nMontants négatifs restants : {(payments['amount_cents'] < 0).sum()}")


Montants négatifs restants : 0


### 5.4 Nettoyage — `status`

**Problème** : 10 variantes pour 6 états réels (casse + typo)  
Mapping : `succeeded/Succeeded/success/suceeded` → `succeeded`  
`failed/FAILED` → `failed`

In [57]:
payments["status"] = (
    payments["status"]
    .str.strip()
    .str.lower()
    .replace({"success": "succeeded", "suceeded": "succeeded"})
)

print(payments["status"].value_counts())

status
succeeded    3980
failed       1984
pending       581
refunded      375
disputed      216
canceled      141
Name: count, dtype: int64


### 5.5 Nettoyage — `currency`

In [58]:
payments["currency"] = (
    payments["currency"]
    .str.strip()
    .str.upper()
    .replace({"": "EUR"})
)

print(payments["currency"].value_counts())

currency
EUR    6748
USD     380
GBP     149
Name: count, dtype: int64


### 5.6 Suppression des orphelins

**Problème** : user_id invalides (0, -1, > 2001) et subscription_id inexistants  
**Décision** : exclusion dans `payments_clean` — ces lignes n'ont pas de user ni subscription traçables

In [59]:
payments["user_missing"] = (~payments["user_id"].isin(users["id"])).astype(int)
print(f"Payments avec user invalide : {payments['user_missing'].sum()}")
print(payments[payments["user_missing"] == 1]["user_id"].value_counts())

# Création du dataset propre
payments_clean = payments[payments["user_missing"] == 0].copy()
payments_clean = payments_clean[
    payments_clean["subscription_id"].isin(subscriptions["id"])
].copy()

print(f"\nPayments conservés : {len(payments_clean)} / {len(payments)}")
print(f"Payments exclus    : {len(payments) - len(payments_clean)}")

Payments avec user invalide : 30
user_id
 0       12
-1        9
 2155     1
 2174     1
 2108     1
 2069     1
 2034     1
 2078     1
 2146     1
 2123     1
 2067     1
Name: count, dtype: int64

Payments conservés : 7126 / 7277
Payments exclus    : 151


### 5.7 paiements = 0

**Observation** : 10 paiements avec `amount_cents=0` et `status=succeeded`  
**Analyse** : ces users ont des paiements normaux après → je suppose que c des free trial 
**Décision** : on conserve ces lignes, elles sont valides métier

In [60]:
free_trials = payments_clean[
    (payments_clean["amount_cents"] == 0) & 
    (payments_clean["status"] == "succeeded")
]
print(f"Free trials identifiés : {len(free_trials)}")

Free trials identifiés : 10


---
## 6. Exploration & Nettoyage — Table `complaints`

### 6.1 Exploration initiale

In [61]:
complaints.info()
print(complaints.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1213 entries, 0 to 1212
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id               1213 non-null   int64
 1   reporter_id      1213 non-null   int64
 2   target_id        1213 non-null   int64
 3   subscription_id  1213 non-null   int64
 4   type             1213 non-null   str  
 5   status           1213 non-null   str  
 6   created_at       1213 non-null   str  
 7   resolved_at      513 non-null    str  
 8   resolution       448 non-null    str  
dtypes: int64(4), str(5)
memory usage: 147.6 KB
id                   0
reporter_id          0
target_id            0
subscription_id      0
type                 0
status               0
created_at           0
resolved_at        700
resolution         765
dtype: int64


In [62]:
complaints.head()

,id,reporter_id,target_id,subscription_id,type,status,created_at,resolved_at,resolution
0,1,904,1565,389,billing_issue,open,2023-09-06 08:02:33,NaN,NaN
1,2,1702,600,260,subscription_inactive,escalated,2024-07-30 12:44:00,NaN,NaN
2,3,1361,672,124,Accès refusé,resolved,2022-10-01 21:19:13,2024-01-27 20:51:20,
3,4,19,1706,317,ACCESS_DENIED,RESOLVED,2024-05-08 04:05:41,2024-06-03 03:41:59,refunded
4,5,327,609,289,ACCESS_DENIED,closed,2021-06-11 13:01:44,2021-10-31 06:56:01,refunded


In [63]:
# Status : problème de casse (open/Open/RESOLVED)
print(complaints["status"].value_counts(dropna=False))

# Type : mélange FR/EN et casses différentes
print("\n", complaints["type"].value_counts(dropna=False))

status
in_progress    194
open           192
RESOLVED       171
resolved       169
escalated      165
Open           164
closed         158
Name: count, dtype: int64

 type
other                    153
wrong_credentials        145
subscription_inactive    143
owner_unresponsive       142
ACCESS_DENIED            135
access_denied            134
billing_issue            127
fraud_suspicion          118
Accès refusé             116
Name: count, dtype: int64


### 6.2 Nettoyage — dates

In [64]:
complaints["created_at"]  = pd.to_datetime(complaints["created_at"],  errors="coerce", utc=True)
complaints["resolved_at"] = pd.to_datetime(complaints["resolved_at"], errors="coerce", utc=True)

now = pd.Timestamp.now(tz="UTC")
print(f"created_at NaT            : {complaints['created_at'].isna().sum()}")
print(f"resolved_at NaT           : {complaints['resolved_at'].isna().sum()} (normal — plaintes non résolues)")
print(f"resolved_at avant created : {(complaints['resolved_at'] < complaints['created_at']).sum()}")
print(f"created_at futur          : {(complaints['created_at'] > now).sum()}")
print(f"resolved_at futur         : {(complaints['resolved_at'] > now).sum()}")

created_at NaT            : 0
resolved_at NaT           : 700 (normal — plaintes non résolues)
resolved_at avant created : 19
created_at futur          : 0
resolved_at futur         : 0


### 6.3 Nettoyage — `status`

**Problème** : même valeur en plusieurs casses (`open`, `Open`, `resolved`, `RESOLVED`)  
**Décision** : tout en lowercase

In [65]:
complaints["status"] = complaints["status"].str.strip().str.lower()
print(complaints["status"].value_counts())

status
open           356
resolved       340
in_progress    194
escalated      165
closed         158
Name: count, dtype: int64


### 6.4 Nettoyage — `type`

**Problème** : `access_denied` et `Accès refusé` sont le même type (FR/EN)  
**Décision** : normaliser en lowercase + fusionner les variantes FR

In [66]:
complaints["type"] = (
    complaints["type"]
    .str.strip()
    .str.lower()
    .replace({"accès refusé": "access_denied"})
)

print(complaints["type"].value_counts())

type
access_denied            385
other                    153
wrong_credentials        145
subscription_inactive    143
owner_unresponsive       142
billing_issue            127
fraud_suspicion          118
Name: count, dtype: int64


### 6.5 Nettoyage — `resolution` vide

**Problème** : 78 chaînes vides `""` non détectées par `isnull()`  
**Décision** : remplacer par NaN

In [67]:
complaints["resolution"] = complaints["resolution"].replace("", None)
print(f"resolution NaN total : {complaints['resolution'].isna().sum()}")

resolution NaN total : 843


### 6.6 Anomalie — status `open` avec `resolved_at`

**Problème** : 15 complaints `open` ont une date de résolution — incohérent  
**Décision** : corriger le status en `resolved` car la date fait foi

In [68]:
mask_open_resolved = (
    (complaints["status"] == "open") & 
    (complaints["resolved_at"].notna())
)
print(f"Status open avec resolved_at : {mask_open_resolved.sum()}")

complaints.loc[mask_open_resolved, "status"] = "resolved"
print(f"Après correction : {((complaints['status'] == 'open') & complaints['resolved_at'].notna()).sum()}")

Status open avec resolved_at : 15
Après correction : 0


### 6.7 Anomalie — `reporter_id == target_id`

**Problème** : 32 plaintes où le reporter et la cible sont la même personne  
**Décision** : flag avec `self_complaint=1`

In [69]:
complaints["self_complaint"] = (
    complaints["reporter_id"] == complaints["target_id"]
).astype(int)

print(f"Self complaints : {complaints['self_complaint'].sum()}")

Self complaints : 32


### 6.8 Nettoyage — `subscription_id` orphelins

In [70]:
complaints["subscription_missing"] = (
    ~complaints["subscription_id"].isin(subscriptions["id"])
).astype(int)

print(f"subscription_id orphelins : {complaints['subscription_missing'].sum()}")

subscription_id orphelins : 20


---
## 7. Reverse-engineering des codes `status`

Aucun dictionnaire de données n'est fourni. On déduit le sens des codes en croisant les tables.

### 7.1 `users.status`

In [ ]:
# Taux d'échec paiements et nombre de complaints par status user
payments_clean["is_failed"] = (payments_clean["status"] == "failed")

payment_profile = (
    users.merge(payments_clean, left_on="id", right_on="user_id", how="left")
    .groupby("status_x")
    .agg(
        nb_users=("id_x", "nunique"),
        nb_payments=("id_y", "count"),
        taux_echec=("is_failed", "mean")
    )
    .round(2)
)
print(payment_profile)

In [ ]:
# Complaints par user par status
complaints_profile = (
    users.merge(complaints, left_on="id", right_on="reporter_id", how="left")
    .groupby("status_x")
    .agg(
        nb_users=("id_x", "nunique"),
        nb_complaints=("id_y", "count"),
        complaints_par_user=("id_y", lambda x: round(len(x) / x.nunique() if x.nunique() > 0 else 0, 2))
    )
    .round(2)
)
print(complaints_profile)

In [ ]:
# % d'owners de subscriptions par status
for s in [-1, 0, 1, 2, 3, 4, 99]:
    users_s    = users[users["status"] == s]["id"].tolist()
    nb_owners  = subscriptions["owner_id"].isin(users_s).sum()
    nb_users   = len(users_s)
    print(f"Status {s:3d} : {nb_users} users → {nb_owners} owners ({round(nb_owners/nb_users*100, 1)}%)")

### 7.2 Conclusion — `users.status`

Après croisement avec `payments`, `memberships`, `complaints` et `subscriptions` :

| Code | Hypothèse | Indice principal |
|------|-----------|------------------|
| 0    | Actif standard | Le plus fréquent (1325), comportement normal |
| 1    | Trial / en attente | Comportement similaire au 0 (179) |
| 2    | Suspendu | Comportement similaire au 0 (107) |
| 3    | Inactif | Plus de complaints/user (2.59) |
| 4    | Restreint | Moins de complaints/user (1.92) |
| -1   | Soft-deleted | Moins d'owners (10.5% vs ~20%), a encore des memberships actifs |
| 99   | Erreur système | **Confirmé anomalie** — 47% échecs paiements, 3.29 complaints/user |
| NaN  | Donnée manquante | 25 lignes |

**Les status 0-4 ont des comportements trop similaires pour être distingués avec certitude.**  
Pour les analyses de risque : `status=99` sera traité comme anomalie, `status=-1` comme compte supprimé.  
Les status 0-4 seront utilisés comme variable catégorielle sans interprétation sémantique forcée.

### 7.3 `memberships.status`

Codes déduits en croisant avec `left_at` et `reason` :

| Code | Hypothèse | Indice |
|------|-----------|--------|
| 0 | En attente / invité | 100% sans left_at ni reason (59) |
| 1 | Actif | 97% sans left_at (414/428) |
| 2 | Parti | left_at présent sur 202/218, raisons variées |
| 3 | Suspendu | left_at présent sur 100% des 107 lignes |
| 4 | En attente confirmation | 100% sans left_at ni reason (64) |
| 5 | Expiré | 100% sans left_at ni reason (84) |
| 6 | Banni | left_at sur 100% des 68, raisons fraud/payment_failed |
| 7 | Résilié | left_at sur 100% des 55 lignes |

### 7.4 `subscriptions.status`

| Code | Hypothèse | Indice |
|------|-----------|--------|
| 0 | Actif | Le plus fréquent (229), 406 membres encore présents |
| 1 | Trial | Peu nombreux (46), 73 membres encore là |
| 2 | Suspendu | 68 abonnements, 113 membres encore présents |
| 3 | Résilié | 57 abonnements, 89 membres encore présents — anomalie |

**Anomalie notable** : les abonnements status=2 et 3 ont encore des membres actifs —  
soit le membership_status n'est pas mis à jour, soit ces membres continuent d'accéder malgré la suspension.

---
## 8. Sauvegarde des tables nettoyées

On exporte chaque table dans un fichier CSV dans le dossier `data/`.  
Pour `payments`, on sauvegarde `payments_clean` (sans les 151 payments orphelins).

In [74]:
users.to_csv("../data/users_clean.csv",                 index=False)
subscriptions.to_csv("../data/subscriptions_clean.csv", index=False)
memberships.to_csv("../data/memberships_clean.csv",     index=False)
payments_clean.to_csv("../data/payments_clean.csv",     index=False)
complaints.to_csv("../data/complaints_clean.csv",       index=False)

print(f"✓ users_clean.csv          ({len(users)} lignes)")
print(f"✓ subscriptions_clean.csv  ({len(subscriptions)} lignes)")
print(f"✓ memberships_clean.csv    ({len(memberships)} lignes)")
print(f"✓ payments_clean.csv       ({len(payments_clean)} lignes, {len(payments) - len(payments_clean)} exclus)")
print(f"✓ complaints_clean.csv     ({len(complaints)} lignes)")

✓ users_clean.csv          (2001 lignes)
✓ subscriptions_clean.csv  (400 lignes)
✓ memberships_clean.csv    (1083 lignes)
✓ payments_clean.csv       (7126 lignes, 151 exclus)
✓ complaints_clean.csv     (1213 lignes)
